# The No Free Lunch Theorem

**Topic:** Why No Algorithm Wins Every Time

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the No Free Lunch theorem in plain English and explain its practical implication
- **Explain** why an algorithm that performs well on one dataset structure can fail on another
- **Interpret** which dataset structures favor which algorithm families

> **Tip:** Select **Linear** in the dropdown and see how well logistic regression performs. Then switch to **Circular** and watch that same algorithm struggle while KNN adapts. Try all four dataset shapes before reading the explanation.

---
## How we got here

In **08_gradient_descent.ipynb** we saw how any algorithm trains by minimizing a loss function. But which algorithm should you use? The answer depends entirely on the structure of your data — and that is what the No Free Lunch theorem formalizes.

This notebook sets up the key question that the entire supervised/ folder will answer: each algorithm makes different assumptions, and understanding those assumptions lets you choose wisely.

---
## Why this matters for data science

The No Free Lunch theorem (Wolpert, 1996) proves mathematically that averaged across all possible datasets, every learning algorithm performs the same. No algorithm is universally better than any other.

This sounds like bad news, but it is actually empowering: it means that when you understand the structure of your data (linear? circular? clustered?), you can choose the algorithm whose assumptions match that structure — and it will outperform every other algorithm on that specific problem.

---
## Try it yourself

In [2]:
np.random.seed(42)
from sklearn.linear_model import LogisticRegression

def make_dataset(name):
    if name == "Linear":
        X, y = make_classification(n_samples=200, n_features=2,
            n_informative=2, n_redundant=0, random_state=42)
    elif name == "Circular":
        X, y = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
    elif name == "Moons":
        X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
    else:  # XOR — axis-aligned cuts win here, so the tree gets its moment
        rng = np.random.RandomState(42)
        X = rng.uniform(-3, 3, size=(200, 2))
        y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)
    return X, y

MODELS = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree":       DecisionTreeClassifier(max_depth=3, random_state=42),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
}

out = Output()

dataset_dd = Dropdown(
    options=["Linear", "Circular", "Moons", "XOR"],
    value="Linear",
    description="Dataset:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))

caption = widgets.HTML(layout=widgets.Layout(width="700px"))

def decision_grid(model, X):
    h = 0.08
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    return xx, yy, Z

def panel(name, clf, X, y):
    clf.fit(X, y)
    score = cross_val_score(clf, X, y, cv=5, scoring="accuracy").mean()
    xx, yy, Z = decision_grid(clf, X)

    # Filled decision regions (two-tone), points on top colored by true class.
    region = go.Heatmap(
        x=xx[0], y=yy[:, 0], z=Z, showscale=False,
        colorscale=[[0, PALETTE["secondary"]], [1, PALETTE["primary"]]],
        opacity=0.25, hoverinfo="skip")
    pts0 = X[y == 0]; pts1 = X[y == 1]
    points0 = go.Scatter(x=pts0[:, 0], y=pts0[:, 1], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=6, opacity=0.9,
                    line=dict(width=0.5, color="white")), showlegend=False, hoverinfo="skip")
    points1 = go.Scatter(x=pts1[:, 0], y=pts1[:, 1], mode="markers",
        marker=dict(color=PALETTE["primary"], size=6, opacity=0.9,
                    line=dict(width=0.5, color="white")), showlegend=False, hoverinfo="skip")

    lay = base_layout(title=f"{name}<br>CV acc = {score:.2f}",
        xaxis_title="Feature 1", yaxis_title="Feature 2")
    lay.update(width=300, height=320,
        xaxis=dict(range=[xx.min(), xx.max()]),
        yaxis=dict(range=[yy.min(), yy.max()]))
    fig = go.Figure(data=[region, points0, points1], layout=lay)
    return go.FigureWidget(fig), score

def render(dataset_name):
    X, y = make_dataset(dataset_name)
    widgets_row, scores = [], {}
    for name, clf in MODELS.items():
        w, s = panel(name, clf, X, y)
        widgets_row.append(w)
        scores[name] = s

    best = max(scores, key=scores.get)
    worst = min(scores, key=scores.get)
    gap = scores[best] - scores[worst]

    shape_note = {
        "Linear":   "Classes split by a straight line — the boundary logistic regression is built to draw.",
        "Circular": "One class encircles the other — no straight line can separate them.",
        "Moons":    "Two interleaved crescents — a smooth curved boundary, not a straight or boxy one.",
        "XOR":      "Diagonal checkerboard — separable only by axis-aligned cuts, the tree's native move.",
    }[dataset_name]

    caption.value = (
        f"<div style='font-size:13px; line-height:1.5; padding:6px 2px'>"
        f"<b>{dataset_name}:</b> {shape_note}<br><br>"
        f"On this shape <b>{best}</b> wins ({scores[best]:.2f}) and "
        f"<b>{worst}</b> trails ({scores[worst]:.2f}) — a {gap*100:.0f}-point gap from "
        f"the <i>same three algorithms</i>, purely because the data changed. That swing is "
        f"the No Free Lunch theorem in miniature: switch datasets and watch the winner change.</div>")

    with out:
        clear_output(wait=True)
        display(HBox(widgets_row))

def on_change(change): render(dataset_dd.value)
dataset_dd.observe(on_change, names="value")
display(VBox([dataset_dd, out, caption]))
render("Linear")

---
## What's happening?

There is no universally best diet. A low-carb approach works for some people, Mediterranean for others, caloric restriction for others. The best diet depends on your biology, lifestyle, and goals. Same with ML algorithms.

Each algorithm makes an **inductive bias**: an assumption about what kind of function it expects the data to follow.

- **Logistic regression** assumes a linear decision boundary. It excels when classes are separable by a straight line and fails on curves.
- **KNN** makes no assumption about shape — it just memorizes training examples and votes. It adapts to any shape but struggles in high dimensions.
- **Decision trees** assume the boundary can be drawn with axis-aligned cuts. They handle step-function patterns well.

| Algorithm | Wins on | Struggles with | Key assumption |
|---|---|---|---|
| Logistic Regression | Linearly separable data | Nonlinear patterns | Linear decision boundary |
| Decision Tree | Step-function patterns | Smooth curves | Axis-aligned splits |
| KNN (k=5) | Any local structure | High dimensions, large datasets | Local similarity |

---
## Real-world example: Algorithm selection in practice

A credit card fraud team tries three algorithms on their data:
- Logistic regression: 89% accuracy (fraud patterns are partly linear — high amounts at unusual hours)
- Decision tree: 91% accuracy (some clear threshold rules: over $5,000 at 3 AM)
- KNN: 78% accuracy (fraud is rare and in high-dimensional transaction space, distances lose meaning)

The No Free Lunch theorem predicts exactly this: no algorithm is universally best. Knowing the structure of fraud patterns (threshold-like rules) explains why tree-based methods win here.

---
## Key takeaway

> **No algorithm wins on every dataset — but understanding the structure of your data lets you choose the algorithm whose assumptions match that structure, and it will outperform all others on your specific problem.**

---
*Next up: Feature Importance — once you have a model, which inputs actually drove the predictions?*